# Seguimiento a embarques
- V19. 2025-03-07
    - Se conserva el OOR original con formulas y datos generados por el usuario
- V18. 2025-03-03
    - Correccion a la integracion de inventorystagebackup
- V17. 2025-02-26
    - Se agregan cajas separadas del inventorystagebackup
    - Se buscan las columnas en el OOR Old
- V16. 2025-02-24
    - Cambio Cancelada por Cancelled
- V15. 2025-02-24
    - Los resultados de yield reports se agregan a la hoja Trov Daily Status NEW
- V14. 2025-02-23
    - Correccion menor
- V13. 2025-02-19
    - Se concatenan las work orders
    - Se integran fechas de llegada al EDI a partir de un OOR antiguo 
- V12. 2025-02-19
    - Se utiliza ELP Master log como fuente de datos
    - Se agregan columnas y calculos al OOR
- V11. 2025-02-13
    - Revisar columnas de elp master log
- V10. 2025-02-09
    - Correccion en links
    - Se agrega fecha al OOR
- V9. 2025-02-08
    - Correccion al proceso de integracion de envios a ELP
    - El Shipment transactions ya no es acumulable debido a que no se pueden eliminar duplicados
    - Se agrega proceso de integracion de Yield reports (falta determinar como visualizarlos)
    - Se agrega inicio para panel
- V8. 2025-02-06
    - El archivo korrus_Data no es obligatorio
    - Drop Zone es NULL en lugar de blanco
- V7. 2025-02-04
    - Mejoras en el manejo de errores
- V6. 2025-02-03
    - Se integra el archivo de ELP master log si esta seleccionado
    - Correcciones para la primera ejecicion del script
- V5. 2025-02-01
    - Se elimino fecha de reparacion    
- V4. 2025-01-31
    - Correcciones para evitar errores por dataframes vacios
    - Se ajustan las columnas de embarques al paso para que sean los del reporte consolidado actual
- V3. 2025-01-28
    - Se guarda el ancho de columnas
    - Se cambia el orden de algunas columnas
- V2. 2025-01-28
    - Se consolidan los archivos Master Edi, Shipped to Cust y Shipped to ELP
    - Se genera el reporte OOR
- V1. Version inicial

## Seleccionar archivos

In [ ]:
# Imports
import pandas as pd
import os
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import pickle
from datetime import datetime, timedelta, date
from openpyxl import load_workbook
from openpyxl.formula.translate import Translator
from openpyxl.styles import PatternFill
from openpyxl.worksheet.table import TableStyleInfo, Table
from openpyxl.utils.cell import coordinate_from_string, get_column_letter, coordinate_from_string, column_index_from_string
from openpyxl.utils.dataframe import dataframe_to_rows
from tkinter import Tk, filedialog as fd
import win32com.client
import warnings
import shutil
from copy import copy

warnings.filterwarnings("ignore")
def open_file_selection(initialdir='',filter_name='*.*'):
    """
    Opens a file selection dialog and returns the selected file paths.
    
    :param initialdir: The initial directory to open in the dialog.
    :return: A tuple of selected file paths or an empty tuple if no file is selected.
    """
    # Create a hidden root window
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)  # Ensure the dialog appears on top
    
    filetypes = (
        (filter_name,f"*{filter_name.split(' ')[0]}*.*"),
        ('All files', '*.*'),
        ('Excel', '*.xlsx'),
        ('CSV', '*.doc')
    )
    files = fd.askopenfilenames(
        filetypes=filetypes,
        initialdir=initialdir
    )
    
    # Destroy the root window after use
    root.destroy()
    
    return files
def select_directory(initialdir):
    """
    Opens a directory selection dialog and returns the selected directory path.
    
    :param initialdir: The initial directory to open in the dialog.
    :return: The selected directory path as a string, or an empty string if no directory is selected.
    """
    # Create a hidden root window
    root = Tk()
    root.withdraw()
    root.attributes('-topmost', True)  # Ensure the dialog appears on top

    # Open the directory selection dialog
    directory = fd.askdirectory(initialdir=initialdir)

    # Destroy the root window after use
    root.destroy()

    return directory

# Function to show a pop-up message
def show_popup_message(message):
    display(Markdown(f"### **{message}**"))

# Decorator to handle the permission error
def handle_permission_error_with_popup(func):
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except PermissionError as e:
            if e.errno == 13:  # Permission denied error
                show_popup_message(f"Error: {e}\nFavor de cerrar el archivo.")
    return wrapper

def save_df(df, filepath, sheet_name='Sheet', index=False):
    with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index)

@handle_permission_error_with_popup
def save_df_multiple(df_dict=dict(), filepath='',index=False):
    with pd.ExcelWriter(filepath) as writer:
        for key in df_dict.keys():
            df_dict[key].to_excel(writer, sheet_name=key, index=index)

@handle_permission_error_with_popup
def save_wb(wb, filepath):
    wb.save(filepath)
    wb.close()

@handle_permission_error_with_popup
def append_sheet(df,path,sheet_name,index):
    with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=index) 

@handle_permission_error_with_popup
def append_df_multiple(df_dict, filepath, index=False):
    with pd.ExcelWriter(filepath, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        for sheet_name, df in df_dict.items():
            df.to_excel(writer, sheet_name=sheet_name, index=index)

def is_file_open(filepath):
    # Check if the file exists
    if not os.path.exists(filepath):
        return False  # File does not exist, so treat it as "open" for your logic

    try:
        # Try to open the file in write mode
        with open(filepath, 'a'):
            pass
        return False  # File is not open (no exception raised)
    except PermissionError:
        return True 
    
def get_path(file_selectors,selector_name):
    file_selector=[file_selector for file_selector in file_selectors if file_selector.children[0].description[0:-1]==selector_name]
    if len(file_selector)==0:
        show_popup_message(f"No se encontro el selector: {selector_name}")
        raise SystemExit()
    file_selector=file_selector[0]
    if not file_selector.children[1].value:
        return None
    selected_path=file_selector.children[1].value.strip()
    return selected_path

def read_excel(path=None,sheet_name=0,header=0):
    with pd.ExcelFile(path) as xls:
        df=pd.read_excel(path,sheet_name=sheet_name,header=header)
    return df

def verify_selections(file_selectors):
    not_selected=[]
    for file_selector in file_selectors:
        selector=file_selector.children[0].description[0:-1]
        selected=file_selector.children[1].value.strip()
        if not os.path.exists(selected):
            file_selector.children[1].value='Not selected'
            selected='Not selected'
        if selector not in mandatory_selectors:
            continue
        if selected=='Not selected':
            not_selected.append(selector)
            continue

    if len(not_selected)>0:
        show_popup_message(f'Favor de seleccionar los siguientes archivos:{not_selected}')
        raise SystemExit()

def close_xl_if_open(path):
    if is_file_open(path):
        try:
            excel = win32com.client.Dispatch("Excel.Application")
            workbook = excel.Workbooks(path)
            workbook.Save()
            workbook.Close()
        except:
            show_popup_message(f"Cerrar el archivo: {path}")
            raise SystemExit()               

def format_dates(df, date_cols=[],type='iso'):
    for col in date_cols:
        if not col in df.columns:
            continue
        if type=='iso':
            df[col]=pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')
        else:
            df[col]=pd.to_datetime(df[col], errors='coerce')
    return df

def format_xl_dates(wb,sheet_name='',date_columns=[]):
    """
    Set format to a date column in an excel worksheet
    """
    ws=wb[sheet_name]
    for col in date_columns:
        col_info=get_column_info(ws,col)
        for cell in col_info['data_range']:
            cell.number_format = 'mm/dd/yyyy'  
    return wb

def  preserve_info(new_dict_wb,path_old_wb,wb_preserve_info={}):
    """
    Preserve the data in columns from an existing workbook in a dict dataframe
    Opens the old workbook and passes the columns to tne new one
    """
    dict_wb = read_excel(path_old_wb,sheet_name=None)
    for wb_key in dict_wb.keys():      
        if not wb_key in wb_preserve_info.keys():
            continue
        user_cols = [item for item in wb_preserve_info[wb_key]['user_cols'] if item in dict_wb[wb_key].columns]
        if not user_cols:
            continue
        df=dict_wb[wb_key][wb_preserve_info[wb_key]['keys']+user_cols]
        df.drop_duplicates(inplace=True)
        new_dict_wb[wb_key]=new_dict_wb[wb_key].merge(df,on=wb_preserve_info[wb_key]['keys'],how='left') 
    return new_dict_wb   

def fillna_by_col(df,columns=[],fill_value=0):
    for col in columns:
        df[col]=df[col].astype(str)
        df[col]=df[col].fillna(fill_value)
    return df

def append_df_to_excel(df_new=pd.DataFrame(),path='',sheet_name='Sheet1',keys=[],date_cols=[]):
    """
    Appends a dataframe to an existing Excel file,
    keys: Fields that should not be duplicated, the old records are kept
    path: Path to the Excel file
    """
    if df_new.empty:
        return
    if not keys:
        keys=df_new.columns
    df_new=df_new.copy()
    df_new=format_dates(df_new,date_cols)
    if not os.path.exists(path):
        # If the file doesn't exist, save the dataframe as a new Excel file
        save_df(df_new,path,sheet_name=sheet_name,index=False)
        return
    close_xl_if_open(path)
    df=read_excel(path)
    df=format_dates(df,date_cols)
    merged = pd.merge(df_new, df[keys], on=keys, how='left', indicator=True)
    df_new=df_new.iloc[merged[merged['_merge'] == 'left_only'].index]
    df=pd.concat([df,df_new])  
    df_grp=df.groupby(keys).count()
    df_grp=df_grp[df_grp[df_grp.columns[0]]>1]
    if len(df_grp)>0:
        df_grp.reset_index(inplace=True)
        show_popup_message(f"Hay duplicados en el archivo {path} para la llave {keys}: {df_grp.head()[keys]}")
        raise SystemExit()
    save_df(df,path,sheet_name=sheet_name,index=False)

def append_df_to_df(df_new=pd.DataFrame(),df_old=pd.DataFrame(),table='',keys=[],date_cols=[]):
    """
    Appends a dataframe to an existing Dataframe
    keys: Fields that should not be duplicated, the old records are kept
    table: Table which is source of the dataframe, just to show it in the error
    date_cols: Columns transformed to date
    """
    if df_new.empty:
        return
    if not keys:
        keys=df_new.columns
    df_new=df_new.copy()
    df_new=format_dates(df_new,date_cols)
    df_old=format_dates(df_old,date_cols)
    df_old['composite_key'] = list(zip(*(df_old[col] for col in keys)))
    df_new['composite_key'] = list(zip(*(df_new[col] for col in keys)))
    df_new=df_new[~df_new['composite_key'].isin(df_old['composite_key'])]
    df_old=pd.concat([df_old,df_new])  
    df_old.drop(columns=['composite_key'],inplace=True)
    df_old.reset_index(inplace=True,drop=True)
    df_grp=df_old.groupby(keys).count()
    df_grp=df_grp[df_grp[df_grp.columns[0]]>1]
    if len(df_grp)>0:
        df_grp.reset_index(inplace=True)
        show_popup_message(f"Hay duplicados en el archivo {table} para la llave {keys}: {df_grp.head()[keys]}")
        raise SystemExit()
    return df_old

def update_dataframe(df_original,df_to_integrate,key_cols=[]):
    """
    Update a dataframe with the values of another dataframe
    key_cols: an array of columns that will be used to match the rows between the dataframes
    data from the secod dataframe is appended if not found in the first one
    """
    df1_indexed = df_original.set_index(key_cols)
    df2_indexed = df_to_integrate.set_index(key_cols)

    # Update matching rows for common columns
    common_cols = df1_indexed.columns.intersection(df2_indexed.columns)
    df1_indexed.update(df2_indexed[common_cols])

    # Append rows from df2 that are not in df1
    new_rows = df2_indexed.loc[~df2_indexed.index.isin(df1_indexed.index)]
    df_original = pd.concat([df1_indexed, new_rows]).reset_index()
    return df_original

##### Excel management functions
def find_cell_by_text(ws, text):
    """
    Find the cell containing the specified date in the worksheet.
    
    :param ws: The worksheet object from openpyxl
    :param date_value: The date value to search for (datetime object or string)
    :return: The cell address (e.g., 'B2') or None if not found
    """
    # for row in ws.iter_rows():
    for row in ws[ws.calculate_dimension()]:
        for cell in row:
            if cell.value == text:
                return cell.coordinate  # Return cell address (e.g., 'B2')
    return None 

def get_cell_properties(cell):
    properties = {}
    fill = cell.fill
    properties["background_color"] = fill
    properties["font"]=cell.font
    properties["alignment"] = cell.alignment
    return properties

def format_cell(cell,properties):
    "Apply properties based on the dict of prooperties"
    cell.fill=copy(properties['background_color'])
    cell.font = copy(properties["font"])
    cell.alignment = copy(properties["alignment"])

def format_on_change(zip_cols, ws, start_row=1, format1=None, format2=None):
    """
    Formats rows in a worksheet based on changes in values across zipped columns.

    Parameters:
    - zip_cols: zip of multiple columns (e.g., zip(col_info1, col_info2, ...)).
    - worksheet: The active worksheet to apply formatting.
    - start_row: Starting row number (default is 1).
    - format1: First alternating format (default yellow).
    - format2: Second alternating format (default light blue).
    """
    # Default styles
    if format1 is None:
        format1 = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")  # Yellow
    if format2 is None:
        format2 = PatternFill(start_color="ADD8E6", end_color="ADD8E6", fill_type="solid")  # Light Blue

    # Initialize tracking variables
    previous_values = None
    current_format = format1

    # Iterate through the zipped columns and their row numbers
    for row_idx, cells_rows in enumerate(zip_cols, start=start_row):
        # Check if the current row values differ from the previous row
        combined_values=""
        for cell in cells_rows:
            combined_values=combined_values+cell.value
        if previous_values != combined_values:
            # Alternate the format
            current_format = format1 if current_format == format2 else format2
            previous_values = combined_values

        # Apply the format to all cells in the current row from the zipped columns
        for col_idx, cell in enumerate(cells_rows):
            # cell = ws.cell(row=row_idx, column=col_idx + 1)
            format_cell(cell, current_format)

def get_column_info(ws, col_name):
    """
    Returns a dictionary with the column cell and the data range for the column.
    """
    col_cell=find_cell_by_text(ws, col_name)
    if not col_cell:
        show_popup_message(f"Column '{col_name}' not found in the sheet.")
        raise SystemExit()
    col_cell=ws[col_cell]
    last_cell=ws[ws.calculate_dimension()][-1][-1].coordinate
    data_range=ws[col_cell.offset(1,0).coordinate:ws.cell(ws[last_cell].row,col_cell.offset(1,0).column).coordinate]
    data_range_list=[cell[0] for cell in data_range]
    return {'data_range':data_range_list,
            'col_cell':col_cell}
   
def get_xl_formatting(table_name=None):
    """
    Returns the formatting for the excel file defined in columns and formatting.xlsx
    return example
    {'oor':
    {
        'columna':{
            'head':{properties},
            'data':{properties}
        }
    }
    }
    """
    wb = load_workbook(output_paths['path_xl_format'])
    ws = wb['column_format']
    col_info=get_column_info(ws,'column_name')
    cell_properties={}
    for cell in col_info['data_range']:
        head_properties=get_cell_properties(cell)
        data_properties=get_cell_properties(cell.offset(0, 1))
        if cell.offset(0, -1).value not in cell_properties:
            cell_properties[cell.offset(0, -1).value]={} #Table name
        if cell.value not in cell_properties[cell.offset(0, -1).value]:
            cell_properties[cell.offset(0, -1).value][cell.value]= {} #Column name
        cell_properties[cell.offset(0, -1).value][cell.value]['head_properties']=head_properties
        cell_properties[cell.offset(0, -1).value][cell.value]['data_properties']=data_properties
    ws = wb['special_format']
    col_info=get_column_info(ws,'format_name')
    cell_properties['special_format']={}
    for cell in col_info['data_range']:
        cell_properties['special_format'][cell.value]=get_cell_properties(cell)

    if table_name:
        cell_properties=cell_properties[table_name]
    return cell_properties  

def get_col_sizes(wb):
    """
    Obtains a dictionary with the column sizes of each sheet in a workbook
    """
    col_sizes={}
    for sheet in wb.sheetnames:
        col_sizes[sheet]=wb[sheet].column_dimensions
    return col_sizes

def get_table_styles(wb):
    """
    Obtains a dictionary with the table styles of each sheet in a workbook.

    Parameters:
    -----------
    wb : openpyxl.Workbook
        The loaded workbook object.

    Returns:
    --------
    dict
        A dictionary where the keys are sheet names and the values are lists of 
        (table_name, table_style) tuples for each table in the sheet.
    """
    table_styles = {}

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        tables = ws._tables  # List of Table objects in the sheet

        table_info=None
        for table in tables:
            
            if hasattr(tables[table], "displayName"):  # Ensure it's an openpyxl Table object
                style_name = tables[table].tableStyleInfo.name if tables[table].tableStyleInfo else "No Style"
                table_info=style_name
                break
        if table_info:
            table_styles[sheet_name] = table_info  # Store tables for the sheet

    return table_styles


def apply_col_sizes(wb,col_sizes):
    """
    Applies the column sizes obtained by function get_col_sizes
    """
    for sheet in col_sizes.keys():
        if sheet not in wb.sheetnames:
            continue
        ws=wb[sheet]
        for column_letter in col_sizes[sheet].keys():
            if column_letter==0:
                continue
            ws.column_dimensions[column_letter].width=col_sizes[sheet][column_letter].width
    return wb
        

def apply_table_styles(wb, table_styles):
    """
    Applies table styles to the first table in each sheet based on a given dictionary.

    Parameters:
    -----------
    wb : openpyxl.Workbook
        The workbook object where the styles will be applied.
    
    table_styles : dict
        A dictionary where keys are sheet names and values are the desired table style names.
        Example: {'EDI Master': 'TableStyleMedium2'}
    
    Returns:
    --------
    None
    """
    for sheet_name, style_name in table_styles.items():
        if sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            if ws._tables:
                continue
            table_range = ws.calculate_dimension()
            table_name = f"{sheet_name}_Table".replace(" ", "")
            table = Table(displayName=table_name, ref=table_range)
            table.tableStyleInfo = TableStyleInfo(
                name=style_name,
                showFirstColumn=False,
                showLastColumn=False,
                showRowStripes=True,
                showColumnStripes=False,
            )
            ws.add_table(table)
    return wb

def update_worksheet(df,path_wb,sheet_name,formula_columns=[]):
    """
    Rewrites the data in a workbook, from a pandas dataframe
    The dataframe columns and order in the dataframe must be the same as the workbook
    Returns the wb, does not save it
    formula_columns: formula columns that will be translated from the second row to the cells below
    """

    # Step 3: Load Workbook and Worksheet
    wb = load_workbook(path_wb)
    ws = wb[sheet_name]

    # Step 4: Identify column indexes for formulas
    header_row = 1  # Assume first row contains headers
    col_indexes = {cell.value: idx+1 for idx, cell in enumerate(ws[header_row]) if cell.value}

    formula_col_indexes = [col_indexes[col] for col in formula_columns if col in col_indexes]

    # Step 5: Extract the first row of formulas (dynamic references)
    formula_templates = {}
    for col_idx in formula_col_indexes:
        formula_cell = ws.cell(row=2, column=col_idx)  # First data row (row 2, assuming headers in row 1)
        if formula_cell.data_type == 'f':  # Ensure it's a formula
            formula_templates[col_idx] = formula_cell.value  # Save original formula

    # Step 6: Clear existing data while preserving formulas
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, max_col=ws.max_column):
        for cell in row:
            if cell.column not in formula_col_indexes:  # Only clear non-formula columns
                cell.value = None

    # Step 7: Write updated DataFrame back to worksheet, avoiding formula columns
    for row_idx, row in df.iterrows():
        for col_idx, value in enumerate(row, start=1):  # start=1 to match Excel indexing
            if col_idx not in formula_col_indexes:
                ws.cell(row=row_idx+2, column=col_idx, value=value)

    # Step 8: Copy formulas down dynamically using Translator
    last_data_row = len(df) + 1  # Adjust for headers
    for col_idx, formula_template in formula_templates.items():
        for row in range(2, last_data_row+1):  # Start from row 2 (first data row)
            translated_formula = Translator(formula_template, origin=f"{ws.cell(row=2, column=col_idx).coordinate}") \
                .translate_formula(f"{ws.cell(row=row, column=col_idx).coordinate}")
            ws.cell(row=row, column=col_idx).value = f"={translated_formula}"  # Assign translated formula

    # Step 9: Save the modified workbook
    return wb

def update_xl_column(ws,df,col_name,):
    """
    Updates an Excel worksheet column with values from a DataFrame.
    
    If the column `col_name` does not exist in the worksheet, it is added as the last column.
    The function then updates the existing or newly created column with values from the corresponding column in `df`.
    df must have the same number of rows and order as the ws so read df and wb together and do not change the order in the df
    """
    if not find_cell_by_text(ws,col_name):
        last_col = ws.max_column + 1
        ws.cell(row=1, column=last_col, value=col_name)
    col_info=get_column_info(ws,col_name)
    for cell,df_value in zip(col_info['data_range'],df[col_name].to_list()):
        cell.value=df_value
    return ws

def get_locations(df,ws,column):
    """
    Obtains all the cell locations in a worksheet (ws) for each text in a column of a dataframe
    Pass a dataframe with 
    """
    coord={}
    for text in df[column].drop_duplicates():
        coord[text]=find_cell_by_text(ws,text)
        coord
    return coord

def get_worksheed_df(ws, key_text):
    """
    This is similar to pd.read_excel, except that the dataframe includes data for the cell coordinates and properties.
    It takes time to read so it is used only for wb where we need to preserve data, cell format and structure.
    Returns a dictionary with a dataframe (including cell properties), header properties, formula properties
    """
    header_row_idx = None
    for col in ws.iter_cols(min_row=ws.min_row, max_row=ws.max_row):
        for cell in col:
            if cell.value is not None and key_text in str(cell.value):
                header_row_idx = cell.row
                break
        if header_row_idx is not None:
            break

    if header_row_idx is None:
        show_popup_message(f"No se encontro el header: {key_text}")
        raise SystemExit()     

    header = [cell for cell in ws[header_row_idx]]
    num_cols = len(header)
    
    formula_dict = {}
    first_data_rows = list(ws.iter_rows(min_row=header_row_idx + 1, max_row=header_row_idx + 1, max_col=num_cols))
    if first_data_rows:
        first_data_row = first_data_rows[0]
        for i, cell in enumerate(first_data_row):
            col_name = header[i].value
            if isinstance(cell.value, str) and cell.value.startswith('='):
                formula_dict[col_name] = cell.value

    data_rows = []
    cell_properties_dict={}
    for row in ws.iter_rows(min_row=header_row_idx + 1, max_row=ws.max_row, max_col=num_cols):
        row_dict = {}
        for i, cell in enumerate(row):
            col_name = header[i].value
            row_dict[col_name] = cell.value
            
            
            if cell.fill.patternType:
                cell_properties=get_cell_properties(cell)
                cell_properties_dict[cell.row]={cell.coordinate:cell_properties}
            else:
                cell_properties=None
            row_dict[f"{col_name}_cell"] = {"coordinate": cell.coordinate,
                                            "properties": cell_properties}
        data_rows.append(row_dict)
    df = pd.DataFrame(data_rows)
    
    header_dict = {}
    for cell in header:
        header_dict[cell.value] = {"coordinate": cell.coordinate,
                                   "properties": get_cell_properties(cell)}
    
    worksheed_df = {
        "df": df,
        "header": header_dict,
        "formulas": formula_dict,
        "properties":cell_properties_dict
    }
    return worksheed_df

def update_sheet(wb_dict, ws):
    df = wb_dict["df"]
    header_info = wb_dict["header"]
    formulas = wb_dict.get("formulas", {})
    properties_info = wb_dict.get("properties", {})

    # Get header row number from first header coordinate
    first_header = next(iter(header_info.values()))
    _, header_row = coordinate_from_string(first_header["coordinate"])
    # Delete all rows below the header
    if ws.max_row > header_row:
        ws.delete_rows(header_row + 1, ws.max_row - header_row)

    # Reorder dataframe columns:
    data_columns = [col for col in df.columns if col is not None and not col.endswith('_cell')]
    format_columns = [col for col in df.columns if col is not None and col.endswith('_cell')]
    header_columns = sorted(
        [col for col in header_info.keys() if col in data_columns],
        key=lambda x: column_index_from_string(coordinate_from_string(header_info[x]['coordinate'])[0])
    )
    remaining_columns = [col for col in data_columns if col not in header_columns]
    ordered_columns = header_columns + remaining_columns
    df_ordered = df[ordered_columns]
    
    # Write dataframe starting below the header
    # start_data_row = header_row + 1
    # for i, r in enumerate(dataframe_to_rows(df_ordered, index=False, header=False)):
    #     ws.append(r)
    #     current_row = start_data_row + i
        # fmt_col = f"{col}_cell"
        # if not (fmt_col in df.columns):
        #     continue
        # properties = properties_info.get(fmt_col['coordinate'])
        # if properties:
        #     cell = ws.cell(row=current_row, column=j)
        #     format_cell(cell,properties)

            
        # for j, col in enumerate(ordered_columns, start=1):
        #     fmt_col = f"{col}_cell"
        #     if fmt_col in df.columns:
        #         fmt_info = df.iloc[i][fmt_col]
        #         if isinstance(fmt_info, dict):
        #             properties = fmt_info.get('properties', {})
        #             if properties:
        #                 cell = ws.cell(row=current_row, column=j)
        #                 format_cell(cell,properties)

    # Write formulas and copy them down using Translator
    for col, formula in formulas.items():
        cell_col = f"{col}_cell"
        if cell_col not in df.columns:
            continue
        first_cell_coord = df.iloc[0][cell_col]['coordinate']
        col_letter, _ = coordinate_from_string(first_cell_coord)
        origin_coord = f"{col_letter}{header_row + 1}"
        for idx, row in df.iterrows():
            dest_coord = f"{col_letter}{header_row + 1 + idx}"
            if idx == 0:
                ws[dest_coord].value = formula
            else:
                ws[dest_coord].value = Translator(formula, origin=origin_coord).translate_formula(dest_coord)
    return ws

def create_new_sheet(wb,sheet_name='Sheet1',df=pd.DataFrame()):
    """
    Creates a new sheet in a workbook and appends a dataframe's data
    """
    if sheet_name in wb.sheetnames:
        wb.remove(wb[sheet_name])
    ws=wb.create_sheet(sheet_name)
    for row in dataframe_to_rows(df, index=False, header=True):
        ws.append(row)
    return wb

# -------------------------

# Save and load state using pickle
def save_state_pickle(state, filename='folder_state_local_shipments.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(state, f)

def load_state_pickle(filename='folder_state_local_shipments.pkl'):
    try:
        with open(filename, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        return {"folder_input": None, "folder_output": None, "selections": {}}

# Agregar las columnas criticas por archivo
filters = [
           'OOR',
           'Tracker',
           'ELP Master',
        #    'Request tracker',
           'Shipment transactions',
           'InventoryStageBakup']


mandatory_cols={
    'Korrus':[
        'PurchaseOrder',
        'PODate',
        'REF02_ClearTextClause',
        'REF02_CustomerOrderNumber',
        'REF02_CarrierAccount',
        'TermsTypeCode',
        'TermsBasisDateCode',
        'description',
        'Routing',
        'JobName',
        'JobNameNotes',
        'ShipmentMarkings',
        'ShipmentMarkingsNotes',
        'ShipToParty',
        'ShipToAddress1',
        'ShipToAddress2',
        'ShipToCity',
        'ShipToState',
        'ShipToPostalCode',
        'ShipToCountryCode',
        'BillToParty',
        'Address1',
        'Address2',
        'BillToCity',
        'BillToState',
        'BillToPostalCode',
        'BillToCountryCode',
        'Supplier',
        'SupplierAddress1',
        'SupplierAddress2',
        'SupplierCity',
        'SupplierState',
        'SupplierPostalCode',
        'SupplierCountryCode',
        'LineNumber',
        'Quantity',
        'Uom',
        'price',
        'ProductServiceID',
        'StorageLocation',
        'AssignedDropZone'
        ],
    'Tracker':[
        'PO cliente',
        'Modelo',
        'WO\n QTY',
        'START DATE',
        'REPROGRAMACION',
        'FINISH DATE',
        'SHIP DATE'
        ],
    'Shipment transactions':[
        'Part No.',
        'Customer PO#',
        'Date Shipped',
        'Qty. Shipped'
    ],
    'InventoryStageBakup':[
        'Producto',
        'PO',
        'Box Id',
        'Cantidad'
    ],
    'ELP Master':[
        'CUU ship Date', 
        'PN', 
        'PO', 
        'DZ', 
        'BOX ID', 
        'Unit\nQTY'],
    'OOR':[
        'PurchaseOrder', 
        'ProductServiceID', 
        'LineNumber', 
        'EDI Received']       
                }

mandatory_selectors=['Tracker','ELP Master','OOR']

families={'TROV':['L','CS-L','CC-L'],
          'RISE':['F','CS-F'],
          'Accesorio':['RISE','M','CBL','LENS','WMA'],
          'DZ':['AssignedDropZone']}

column_mapping={
    'PO':['Customer PO#','PO cliente','PurchaseOrder','PO.'],
    'Modelo':['Producto','ProductServiceID','Part No.','PN','ProductService ID'],
    'Quantity':['WO\n QTY','Cantidad','PO Quantity','QTY','WO\n QTY','Unit\nQTY'],
    'ShipmentDate':['CUU ship Date'],
    'PODate':['EDI Received'],
    'WO':['Work Order'],  
    'SOUS': ['REF02_CustomerOrderNumber']  
}

def rename_columns(df, df_col_rel, table_from='std_name', sheet_from='only', table_to='std_name', sheet_to='only'):
    """
    Renames columns of a DataFrame to a standard name or between tables.

    Parameters:
    -----------
    df : pd.DataFrame
        The DataFrame whose columns need renaming.
    df_col_rel : pd.DataFrame
        DataFrame containing the column relationships with the columns:
        ['table', 'sheet', 'column_name', 'std_name'].
    table_from : str
        The source selector to map from.
    sheet_from : str
        The source sheet name.
    selector_to : str
        The target selector to map to.
    sheet_to : str
        The target sheet name.

    Returns:
    --------
    pd.DataFrame
        DataFrame with renamed columns.

    Raises:
    -------
    ValueError:
        If required columns are missing or if no matching mapping is found.
    """
    # Validate that df_col_rel contains all required columns
    required_cols = {'table', 'sheet', 'column_name', 'std_name'}
    missing_cols = required_cols - set(df_col_rel.columns)
    if missing_cols:
        raise ValueError(f"df_col_rel is missing required columns: {missing_cols}")
    
    mapping = {}

    if table_to == 'std_name':
        filtered = df_col_rel[(df_col_rel['table'] == table_from) & 
                              (df_col_rel['sheet'] == sheet_from)]
        if filtered.empty:
            raise ValueError("No matching mapping found for given table_from and sheet_from.")
        mapping = filtered.set_index('column_name')['std_name'].to_dict()
    
    elif table_from == 'std_name':
        filtered = df_col_rel[(df_col_rel['table'] == table_to) & 
                              (df_col_rel['sheet'] == sheet_to)]
        if filtered.empty:
            raise ValueError("No matching mapping found for given table_to and sheet_to.")
        mapping = filtered.set_index('std_name')['column_name'].to_dict()
    
    else:
        filtered_from = df_col_rel[(df_col_rel['table'] == table_from) & 
                                   (df_col_rel['sheet'] == sheet_from)]
        if filtered_from.empty:
            raise ValueError("No matching mapping found for given table_from and sheet_from.")
        mapping_from = filtered_from.set_index('column_name')['std_name'].to_dict()
        # First renaming step
        df = df.rename(columns=mapping_from)

        filtered_to = df_col_rel[(df_col_rel['table'] == table_to) & 
                                 (df_col_rel['sheet'] == sheet_to)]
        if filtered_to.empty:
            raise ValueError("No matching mapping found for given table_to and sheet_to.")
        mapping = filtered_to.set_index('std_name')['column_name'].to_dict()

    # Rename using the final mapping and return a new DataFrame
    return df.rename(columns=mapping) 

def standardize_columns(df, column_mapping):
    """Estandariza las columnas de un DataFrame según un diccionario de mapeo."""
    reverse_mapping = {orig: standard for standard, cols in column_mapping.items() for orig in cols}
    return df.rename(columns=reverse_mapping)

def inverse_standardize_columns(df, column_mapping, columns_to_replace):
    """Reverts standard column names to original names based on column_mapping."""
    inverse_mapping = {standard: orig for standard, cols in column_mapping.items() for orig in cols if orig in columns_to_replace}
    return df.rename(columns=inverse_mapping)

def check_mandatory_cols(cols,selector_name, raise_error=True):
    missing_columns = [col for col in mandatory_cols[selector_name] if col not in cols]
    if len(missing_columns)>0:
        show_popup_message(f"No se encontraron las siguientes columnas en el archivo {selector_name}: {missing_columns}")
        if raise_error:
            raise SystemExit()
        return False
    return True

def set_family(df,column,dest_col='Family'):
    """
    column: name of the column that contains the Model
    """
    for key in families.keys():
        df.loc[df[column].str.startswith(tuple(families[key])), dest_col] = key
    df[dest_col].fillna('Other', inplace=True)
    return df

state = load_state_pickle()
if ('folder_output' not in state.keys()):
    state['folder_output']=''



def set_paths(path):
    """
    Set global paths
    """
    global output_paths
    output_paths={}
    output_paths['path_attachments']=os.path.join(path, "attachments")
    output_paths['path_attachments_done']=os.path.join(output_paths['path_attachments'], "Done")
    output_paths['path_korrus_list']=os.path.join(path,'Korrus_list.xlsx')
    output_paths['path_korrus_data']=os.path.join(path,'Korrus_data.xlsx')
    output_paths['path_edi']=os.path.join(path,'EDI Master.xlsx')
    output_paths['path_ship_cust']=os.path.join(path,'Shipped to Cust.xlsx')
    output_paths['path_ship_elp']=os.path.join(path,'Shipped to ELP.xlsx')
    output_paths['path_oor']=os.path.join(path,'OOR Report.xlsx')
    output_paths['path_xl_format']=os.path.join(path,'columns and formatting.xlsx')
    output_paths['path_graph_data']=os.path.join(path,'graph_data.xlsx')

def set_col_rel(output_paths):
    global df_col_rel,df_columns,col_names
    df_columns=read_excel(output_paths['path_xl_format'],sheet_name='column_format')
    df_col_rel=df_columns[~df_columns['std_name'].isnull()].copy()

folder_output_button = widgets.Button(description="Folder de salidas:")
if state['folder_output']:
    initialdir=state['folder_output']
    set_paths(initialdir)
else:
    initialdir='Not selected'
folder_output_label = widgets.Label(value=initialdir)

def on_output_button_click(b):
    if state['folder_output']:
        initialdir=state['folder_output']
    else:
        initialdir='/'
    selected_dir = select_directory(initialdir=initialdir)
    if selected_dir:
        folder_output_label.value = f"{selected_dir}"
        state['folder_output']=selected_dir
        save_state_pickle(state)
    else:
        folder_output_label.value = "Not selected"
    set_paths(folder_output_label.value)




# Attach the event to the folder_output_button
folder_output_button.on_click(on_output_button_click)

# Create an array of button and label widgets
file_selectors = []
for filter_name in filters:
    # Create button and label
    button = widgets.Button(description=f"{filter_name}:")
    if state.get(filter_name):
        label = widgets.Label(value=f" {state.get(filter_name)}")
    else:
        label = widgets.Label(value=f" Not selected")
    
    # Define the button click event
    def on_button_click(filter_name=filter_name, label=label):
        if (filter_name in state.keys()) and (state[filter_name]):
            initialdir=os.path.dirname(state[filter_name][0])
        else:
            initialdir='/'
        selected_dir = open_file_selection(initialdir=initialdir,filter_name=filter_name)  # Adjust initialdir as needed
        if selected_dir:
            label.value = f" {selected_dir[0]}"
            state[filter_name]=selected_dir[0]
            
            
        else:
            label.value = f" Not selected"
            state[filter_name]=f" Not selected"
        save_state_pickle(state)

    # Attach the event to the button
    button.on_click(lambda b, f=filter_name, l=label: on_button_click(f, l))
    
    # Add the button and label as a horizontal box
    file_selectors.append(widgets.HBox([button, label]))


datepicker_email = widgets.DatePicker(
    description='Fecha mail',
    disabled=False,
    value=date.today()-timedelta(days=1)
)

datepicker_shipments_elp = widgets.DatePicker(
    description='ELP Shipments',
    disabled=False,
    value=date.today()
)

checkbox = widgets.Checkbox(
    value=True,
    description='Integrar Korrus Data',
    disabled=False,
    indent=False
)

set_col_rel(output_paths)


folder_output_button = widgets.Button(description="Folder de salidas:")
if state['folder_output']:
    initialdir=state['folder_output']
    set_paths(initialdir)
else:
    initialdir='Not selected'
folder_output_label = widgets.Label(value=initialdir)


def on_output_button_click(b):
    if state['folder_output']:
        initialdir=state['folder_output']
    else:
        initialdir='/'
    selected_dir = select_directory(initialdir=initialdir)
    if selected_dir:
        folder_output_label.value = f"{selected_dir}"
        state['folder_output']=selected_dir
        save_state_pickle(state)
    else:
        folder_output_label.value = "Not selected"
    set_paths(folder_output_label.value)




# Attach the event to the folder_output_button
folder_output_button.on_click(on_output_button_click)

if 'folder_yield_output' not in state:
    state['folder_yield_output'] = ''

folder_yield_initial = state['folder_yield_output'] if state['folder_yield_output'] else 'Not selected'
folder_yield_button = widgets.Button(description="Yield reports:")
folder_yield_label = widgets.Label(value=folder_yield_initial)

def on_yield_button_click(b):
    initialdir = state['folder_yield_output'] if state['folder_yield_output'] else '/'
    selected_dir = select_directory(initialdir=initialdir)
    if selected_dir:
        folder_yield_label.value = f"{selected_dir}"
        state['folder_yield_output'] = selected_dir
        save_state_pickle(state)
    else:
        folder_yield_label.value = "Not selected"

folder_yield_button.on_click(on_yield_button_click)



# Create an array of button and label widgets
file_selectors = []
for filter_name in filters:
    # Create button and label
    button = widgets.Button(description=f"{filter_name}:")
    if state.get(filter_name):
        label = widgets.Label(value=f" {state.get(filter_name)}")
    else:
        label = widgets.Label(value=f" Not selected")
    
    # Define the button click event
    def on_button_click(filter_name=filter_name, label=label):
        if (filter_name in state.keys()) and (state[filter_name]):
            initialdir=os.path.dirname(state[filter_name][0])
        else:
            initialdir='/'
        selected_dir = open_file_selection(initialdir=initialdir,filter_name=filter_name)  # Adjust initialdir as needed
        if selected_dir:
            label.value = f" {selected_dir[0]}"
            state[filter_name]=selected_dir[0]
            
            
        else:
            label.value = f" Not selected"
            state[filter_name]=f" Not selected"
        save_state_pickle(state)

    # Attach the event to the button
    button.on_click(lambda b, f=filter_name, l=label: on_button_click(f, l))
    
    # Add the button and label as a horizontal box
    file_selectors.append(widgets.HBox([button, label]))


datepicker_email = widgets.DatePicker(
    description='Fecha mail',
    disabled=False,
    value=date.today()-timedelta(days=1),
    layout=widgets.Layout(width='350px'),
    style={'description_width': '170px'}
)

datepicker_shipments_elp = widgets.DatePicker(
    description='ELP Shipments',
    disabled=False,
    value=date.today(),
    layout=widgets.Layout(width='350px'),
    style={'description_width': '170px'}
)

def on_date_change(change):
        state['freeze_date']=change['new']
        save_state_pickle(state)

freeze_date=date.today()
if 'freeze_date' in state.keys():
    freeze_date=state['freeze_date']
datepicker_freeze = widgets.DatePicker(
    description='Fecha Inicial de Actualizacion',
    disabled=False,
    value=freeze_date,
    layout=widgets.Layout(width='350px'),
    style={'description_width': '170px'}
)
datepicker_freeze.observe(on_date_change, names='value')

checkbox = widgets.Checkbox(
    value=True,
    description='Integrar Korrus Data',
    disabled=False,
    indent=False
)

checkbox_oor_dates = widgets.Checkbox(
    value=False,
    description='Integrar fechas del OOR',
    disabled=False,
    indent=False
)

# ui = widgets.VBox([datepicker_email,datepicker_shipments_elp,checkbox]+[widgets.HBox([folder_output_button, folder_output_label])]+file_selectors)
ui = widgets.VBox([
    datepicker_email,
    datepicker_shipments_elp,
    datepicker_freeze,
    checkbox,
    checkbox_oor_dates,
    widgets.HBox([folder_output_button, folder_output_label])
    
] + file_selectors + [widgets.HBox([folder_yield_button, folder_yield_label])])
display(ui)
set_col_rel(output_paths)


## Explorar outlook
- Obtiene los archivos Korrus adjuntos de las fechas no obtenidas anteriormente

In [2]:
# Obtener los archivos adjuntos de los emails
# Crear folder para adjuntos si no existe

if not os.path.exists(output_paths['path_attachments']):
    os.makedirs(output_paths['path_attachments'])

outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")
inbox = outlook.GetDefaultFolder(6)  
date_limit = datepicker_email.value
date_limit=datetime.combine(date_limit, datetime.min.time())
messages=list()
for folder in inbox.folders:
    messages=messages+list(folder.Items)
messages=messages+list(inbox.Items)
# messages.Sort("[ReceivedTime]", True)  
for message in messages:
    if not hasattr(message, 'ReceivedTime'):
        continue
    received_time = message.ReceivedTime
    received_time = datetime.fromtimestamp(received_time.timestamp())
    if received_time < date_limit:
        break 
    if message.Attachments.Count > 0:
        for attachment in message.Attachments:
            # Save the attachment
            attachment_name = attachment.FileName
            attachment_name=f"{os.path.splitext(attachment_name)[0]}_{received_time.strftime('%Y-%m-%d-%H%M%S')}{os.path.splitext(attachment_name)[1]}"
            # if attachment_name in df_korrus_list['file_name'].values:
            #     continue
            if 'KORRUS' not in attachment_name.upper():
                continue
            path_attachment = os.path.join(output_paths['path_attachments'], attachment_name)
            if  not os.path.exists(path_attachment):
                try:
                    attachment.SaveAsFile(path_attachment)
                except Exception as e:
                    print(f"Error saving attachment: {e}")


### Consolidar archivos Korrus
En caso de un error, hay que abrir el archivo Korrus_list.xlsx y eliminar la e (error) del archivo que se corrigio
En caso de requerir reprocesar un archivo, hay que abrir el archivo Korrus_list.xlsx y poner r (reproceso) en el archivo que se va a reprocesar

In [3]:
#Crear la lista de archivos descargados para manejar su integracion. Se procesa lo que esta en el folder sin importar su origen

if not os.path.exists(output_paths['path_attachments_done']):
    os.makedirs(output_paths['path_attachments_done'])

#Status maneja el proceso a realizar en los archivos: r=reprocesar, d=done, e=error
if os.path.exists(output_paths['path_korrus_list']):
    df_korrus_list=pd.read_excel(output_paths['path_korrus_list'])
else:
    df_korrus_list=pd.DataFrame(columns=['file_name','received_time','status'])
close_xl_if_open(output_paths['path_korrus_list'])

files=os.listdir(output_paths['path_attachments'])
df_korrus_list_new=pd.DataFrame(columns=['file_name'],data=files)
df_korrus_list_new['received_time']=df_korrus_list_new['file_name'].str.extract(r'(\d{4}-\d{2}-\d{2}-\d{6})')
df_korrus_list_new=df_korrus_list_new.merge(df_korrus_list,how='outer',on=['file_name','received_time'])
df_korrus_list_new.dropna(subset=['received_time'],inplace=True)
df_korrus_list_new['status'].fillna('',inplace=True)

# Analizar archivos adjuntos
df_korrus_data_new=pd.DataFrame(columns=['origin_file']+mandatory_cols['Korrus'])
for index, row in df_korrus_list_new[df_korrus_list_new['status'].isin(['','r'])].iterrows():
    filepath=os.path.join(output_paths['path_attachments'],row['file_name'])
    if (row['status']=='r') or (not os.path.exists(filepath)):
        filepath=os.path.join(output_paths['path_attachments_done'],row['file_name'])
    if not os.path.exists(filepath):
            df_korrus_list_new.loc[index,'status']='e'
            show_popup_message(f"El archivo {row['file_name']} no existe")
            continue
    if os.path.splitext(os.path.basename(filepath))[1].lower()==".xlsx":
        df = pd.read_excel(filepath)
    elif os.path.splitext(os.path.basename(filepath))[1].lower()==".csv":
        try:
            df = pd.read_csv(filepath, sep='\t', encoding='utf-16')
        except:
            df = pd.read_csv(filepath, sep=',')
    else:
        df_korrus_list_new.loc[index,'status']='e'
        show_popup_message(f"El archivo {row['file_name']} no pudo ser procesado")
        continue
    if check_mandatory_cols(df.columns,selector_name='Korrus',raise_error=False):
        df['origin_file']=row['file_name']
        df_korrus_data_new=pd.concat([df_korrus_data_new,df])
        df_korrus_data_new=df_korrus_data_new[['origin_file']+mandatory_cols['Korrus']]
        df_korrus_list_new.loc[index,'status']='n'
    else:
        df_korrus_list_new.loc[index,'status']='e'

# Consolidar datos en un solo archivo
if os.path.exists(output_paths['path_korrus_data']):
    df_korrus_data=pd.read_excel(output_paths['path_korrus_data'])
else:
    df_korrus_data=pd.DataFrame(columns=['origin_file']+mandatory_cols['Korrus'])
close_xl_if_open(output_paths['path_korrus_data'])
df_korrus_data=df_korrus_data[~df_korrus_data['origin_file'].isin(df_korrus_data_new['origin_file'])]
df_korrus_data_new=pd.concat([df_korrus_data, df_korrus_data_new])


In [ ]:
# Mover archivos a la carpeta de archivos procesados
for index,row in df_korrus_list_new[df_korrus_list_new['status']=='n'].iterrows():
    df_korrus_list_new.loc[index,'status']='y'
    if os.path.exists(f'{output_paths['path_attachments']}/{row["file_name"]}'):
        shutil.move(f'{output_paths['path_attachments']}/{row["file_name"]}', f'{output_paths['path_attachments_done']}/{row["file_name"]}')
# Eliminar archivos ya procesados del folder de llegada
df_remove=df_korrus_list_new[(df_korrus_list_new['status']=='y') & (df_korrus_list_new['file_name'].isin(files))]
for index,row in df_remove.iterrows():
    if os.path.exists(os.path.join(output_paths['path_attachments'],row['file_name'])):
        os.remove(os.path.join(output_paths['path_attachments'],row['file_name']))
files=os.listdir(output_paths['path_attachments'])
# Guardar datos y lista de archivos procesados
save_df(df_korrus_list_new,filepath=output_paths['path_korrus_list'],sheet_name='Korrus List',index=False)
save_df(df_korrus_data_new,filepath=output_paths['path_korrus_data'],sheet_name='Korrus Data',index=False)

## Generar reportes
- Hay tres reportes EDI Master, Shipped to Cust, Shipped to ELP
- Si hay archivos seleccionados se integran a estos reportes

### Consolidar
- Korrus_data --> EDI Master
- InventoryStage --> Shipment to ELP
- Shipment transactions: Standalone file


In [25]:
# Consolidar reportes 
verify_selections(file_selectors)
close_xl_if_open(output_paths['path_oor'])
path_ship_elp_master=get_path(file_selectors,'ELP Master')
wb_elp=load_workbook(path_ship_elp_master)
col_sizes_elp_master=get_col_sizes(wb_elp)
table_styles_elp_master=get_table_styles(wb_elp)
if not os.path.exists(output_paths['path_xl_format']):
    show_popup_message("No se encuentra el archivo: columns and formatting.xlsx")
    raise SystemExit()

if folder_output_label.value=='Not selected':
    show_popup_message("Seleccione el folder de salida")
    raise SystemExit()

# Demanda del cliente
if not os.path.exists(output_paths['path_korrus_data']):
    df=pd.DataFrame(columns=[mandatory_cols['Korrus']+['origin_file']])
    save_df(df,filepath=output_paths['path_korrus_data'],sheet_name='Korrus Data',index=False)

df_korrus_data_new=pd.read_excel(output_paths['path_korrus_data'])
if (len(df_korrus_data_new)>0) & (checkbox.value):
    df_korrus_data_new.loc[:, ~df_korrus_data_new.columns.str.startswith('Unnamed:')]
    df_korrus_data_new=df_korrus_data_new[~df_korrus_data_new['PurchaseOrder'].str.contains('---')]
    df_korrus_data_new['PODate']=pd.to_datetime(df_korrus_data_new['PODate'],format='mixed', errors='coerce').dt.strftime('%Y-%m-%d')
    df_korrus_data_new['LineNumber']=df_korrus_data_new['LineNumber'].astype(int)
    key_korrus=['PurchaseOrder','LineNumber','PODate','ProductServiceID','AssignedDropZone']
    df_korrus_data_new.drop_duplicates(subset=key_korrus,keep='last',inplace=True)
    
    # Eliminar registros no requeridos
    df_korrus_data_new['EDI Received']=df_korrus_data_new['origin_file'].str.extract(r'(\d{4}-\d{2}-\d{2})')
    df_korrus_data_new.drop('origin_file',axis=1,inplace=True)
    df_korrus_data_new.drop_duplicates(inplace=True)
    df_korrus_data_new['PODate']=pd.to_datetime(df_korrus_data_new['PODate'])
    df_korrus_data_new.rename({
        'PurchaseOrder':'PO',
        'ProductServiceID':'ProductService ID'
        },axis=1,inplace=True)
    edi_keys=['PO','PODate','LineNumber','ProductService ID','AssignedDropZone']
    col_sizes={}
    df_edi=read_excel(path_ship_elp_master,sheet_name='EDI Master')
    df_edi=append_df_to_df(df_new=df_korrus_data_new,df_old=df_edi,table='EDI Master',keys=edi_keys,date_cols=['PODate'])
    df_edi=format_dates(df_edi,date_cols=['PODate','EDI Received'],type='')
    append_sheet(df_edi,path_ship_elp_master,sheet_name='EDI Master',index=False)
    wb=load_workbook(path_ship_elp_master)
    wb=format_xl_dates(wb,sheet_name='EDI Master',date_columns=['PODate','EDI Received'])
    wb=apply_table_styles(wb,table_styles_elp_master)
    wb=apply_col_sizes(wb,col_sizes_elp_master)
    save_wb(wb,path_ship_elp_master)


In [26]:
# Obtener fechas de llegada del OOR viejo
path_oor_old=get_path(file_selectors,'OOR')
if checkbox_oor_dates.value:
    dict_oor_old=read_excel(path_oor_old,sheet_name=None)
    df_oor_old=pd.DataFrame()
    for sheet in dict_oor_old.keys():
        if not 'OOR' in sheet:
            continue
        df=dict_oor_old[sheet].copy()
        if 'Unnamed' in df.columns[0]:
            header_index = df[df.iloc[:, 1].astype(str).str.contains("EDI Received", na=False)].index[0]
            df.columns = df.iloc[header_index]
            df = df.iloc[header_index + 1:].reset_index(drop=True)
        df_oor_old=pd.concat([df_oor_old,df])
    check_mandatory_cols(df_oor_old.columns,'OOR')
    df_oor_old=rename_columns(df_oor_old,df_col_rel,table_from='OOR Report',sheet_from='OOR',table_to='ELP Master',sheet_to='EDI Master')
    column_edi_rec='EDI Received'
    df_oor_old=df_oor_old[['PO','ProductService ID','LineNumber',column_edi_rec]].drop_duplicates(['PO','ProductService ID','LineNumber'])
    df_edi=read_excel(path_ship_elp_master,sheet_name='EDI Master')
    if not column_edi_rec in df_edi.columns:
        df_edi[column_edi_rec]=''
    df_edi['ProductService ID']=df_edi['ProductService ID'].str.upper()
    df_edi = df_edi.merge(df_oor_old, how='left', on=['PO', 'ProductService ID', 'LineNumber'], suffixes=('_df1', '_df2'),sort=False)
    df_edi[column_edi_rec] = df_edi[f'{column_edi_rec}_df1']
    df_edi.loc[df_edi[column_edi_rec].isnull() | (df_edi[column_edi_rec] == ''), column_edi_rec] = df_edi[f'{column_edi_rec}_df2']
    df_edi.drop(columns=[f'{column_edi_rec}_df1',f'{column_edi_rec}_df2'],inplace=True)
    wb=load_workbook(path_ship_elp_master)
    ws=wb['EDI Master']
    ws=update_xl_column(ws,df_edi,col_name=column_edi_rec)
    save_wb(wb,path_ship_elp_master)
    
# Shipment transactions, lo embarcado al cliente
path_ship_cust_new=get_path(file_selectors,'Shipment transactions')
if path_ship_cust_new!='Not selected':
    df_ship_cust_new=pd.read_excel(path_ship_cust_new)
    check_mandatory_cols(df_ship_cust_new.columns,'Shipment transactions')
    df_ship_cust_new=df_ship_cust_new[~df_ship_cust_new['Customer PO#'].isna()]
    # append_df_to_excel(df_ship_cust_new,output_paths['path_ship_cust'],sheet_name='Shipped to Cust',keys=['Part No.', 'Customer PO#', 'Date Shipped'])
    # Shipment transactions is not agregated, it is replaced
    save_df(df_ship_cust_new,output_paths['path_ship_cust'],sheet_name='Shipped to Cust',index=False)

# InventoryStage, lo que se embarco a ELP 
path_ship_elp_new=get_path(file_selectors,'InventoryStageBakup')
if path_ship_elp_new!='Not selected':
    df_ship_elp_new=read_excel(path_ship_elp_new)
    check_mandatory_cols(df_ship_elp_new.columns,'InventoryStageBakup')
    df_ship_elp_new=df_ship_elp_new[df_ship_elp_new['Cliente']!='Total']
    df_ship_elp_new=df_ship_elp_new[['Cliente','PO','Producto','Box Id','Cantidad']].ffill()
    df_ship_elp_new=df_ship_elp_new[~df_ship_elp_new['PO'].isna()]
    df=df_ship_elp_new['PO'].str.split('|', expand=True)
    df_ship_elp_new['DZ']=''
    if len(df.columns)>1:
        df_ship_elp_new['DZ']=df[1].str.strip()
        df_ship_elp_new['PO']=df[0].str.strip()
    df_ship_elp_new['CUU ship Date']=datepicker_shipments_elp.value
    df_ship_elp_new['CUU ship Date']=pd.to_datetime(df_ship_elp_new['CUU ship Date']).dt.strftime('%m/%d/%Y')
    df_ship_elp_new['BOX qty']=1
    df_ship_elp_new=df_ship_elp_new.groupby(['PO','Producto','DZ','Box Id']).agg({
        'Cantidad': 'sum',
        'BOX qty':'count',
        'CUU ship Date':'last'
    }).reset_index()
    df_ship_elp_new=rename_columns(df_ship_elp_new,df_col_rel,table_from='InventoryStageBakup',table_to='ELP Master',sheet_to='Shipment to ELP')
    df_ship_elp_master=read_excel(path_ship_elp_master,sheet_name='Shipment to ELP')
    df_ship_elp_master['DZ'].fillna('NULL',inplace=True)
    df_ship_elp_new['DZ'].fillna('NULL',inplace=True)
    df_ship_elp_new.loc[df_ship_elp_new['DZ']=='NA','DZ']='NULL'
    df_ship_elp_master=append_df_to_df(df_new=df_ship_elp_new,df_old=df_ship_elp_master,table='Shipment to ELP',keys=['PO','PN','BOX ID','DZ'])
    df_ship_elp_master['Producto'].fillna('',inplace=True)
    df_ship_elp_master['CUU ship Date']=pd.to_datetime(df_ship_elp_master['CUU ship Date'], errors='coerce').dt.strftime('%m/%d/%Y')
    df_ship_elp_master=set_family(df_ship_elp_master,column='PN',dest_col='Producto')
    df_edi=read_excel(path_ship_elp_master,sheet_name='EDI Master')
    df_po_data=df_edi.drop_duplicates(subset=['PO'],keep='first')
    df_po_data=rename_columns(df_po_data,df_col_rel,table_from='ELP Master',sheet_from='EDI Master',table_to='ELP Master',sheet_to='Shipment to ELP')
    df_ship_elp_master=df_ship_elp_master.drop(columns=['PODate']).merge(df_po_data[['PO', 'PODate']], on='PO', how='left')
    elp_cols=df_columns[df_columns['sheet']=='Shipment to ELP']['column_name'].tolist()
    elp_cols=[x for x in df_ship_elp_master.columns if x in elp_cols]
    df_ship_elp_master=df_ship_elp_master[elp_cols]
    df_ship_elp_master['CUU ship Date']=pd.to_datetime(df_ship_elp_master['CUU ship Date'],errors='coerce')
    append_sheet(df_ship_elp_master,path_ship_elp_master,sheet_name='Shipment to ELP',index=False)
    wb=load_workbook(path_ship_elp_master)
    wb=apply_table_styles(wb,table_styles_elp_master)
    wb=apply_col_sizes(wb,col_sizes_elp_master)
    ws=format_xl_dates(wb,sheet_name='Shipment to ELP',date_columns=['CUU ship Date','PODate'])
    save_wb(wb,path_ship_elp_master)

In [27]:
# Ordenes Canceladas

def cancel_orders(df=pd.DataFrame(),df_cancelled=pd.DataFrame(),table_from='',sheet_from='',cancel_col=''):
    """
    Returns the inpt dataframe with the value 'Cancelled' in column defined by cancel_col
    both Dataframes must have columns corresponding to PO and Modelo 
    """
    df=rename_columns(df,df_col_rel,table_from=table_from,sheet_from=sheet_from)
    df=df.merge(df_cancelled,how='left',on=['PO','Modelo'])
    df.loc[(~df['status_cancelled'].isnull()) & 
           (df[cancel_col].isna() | 
            (df[cancel_col]=='') | 
            (df[cancel_col]==0)), cancel_col] = 'Cancelled'
    df=rename_columns(df,df_col_rel,table_to=table_from,sheet_to=sheet_from)
    df.drop(columns=['status_cancelled'],inplace=True)
    return df

def cancel_orders_in_wb(df,wb,sheet_name,cancel_col):
    indexes=df[df[cancel_col]=='Cancelled'].index
    ws=wb[sheet_name]
    col_info=get_column_info(ws,cancel_col)
    for idx in indexes:
        col_info['data_range'][idx].value='Cancelled'
    return wb
    

df_cancelled=read_excel(path_ship_elp_master,sheet_name='Cancelled Orders')
df_cancelled=rename_columns(df_cancelled,df_col_rel,table_from='ELP Master',sheet_from='Cancelled Orders')
df_cancelled=df_cancelled[['PO','Modelo']].drop_duplicates()
df_cancelled['Modelo']=df_cancelled['Modelo'].str.upper()
df_cancelled['status_cancelled']='cancelled'

In [28]:
# Aplicar ordenes canceladas de la pestaña 'Cancelled Orders'
df_edi=read_excel(path_ship_elp_master,sheet_name='EDI Master')
df_edi=cancel_orders(df_edi,df_cancelled=df_cancelled,table_from='ELP Master',sheet_from='EDI Master',cancel_col='Order/Line cancelled?')

df_ship_elp=read_excel(path_ship_elp_master,sheet_name='Shipment to ELP')
df_ship_elp=cancel_orders(df_ship_elp,df_cancelled=df_cancelled,table_from='ELP Master',sheet_from='Shipment to ELP',cancel_col='Order cancelled?')
wb=wb_elp
cancel_orders_in_wb(df_edi,wb=wb,sheet_name='EDI Master',cancel_col='Order/Line cancelled?')
cancel_orders_in_wb(df_ship_elp,wb=wb,sheet_name='Shipment to ELP',cancel_col='Order cancelled?')
save_wb(wb,path_ship_elp_master)

In [29]:
# Funcion, asignar cantidades
def assign_quantities(df_pos, df_to_assign, additional_fields=[]):
    """
    Asigna las cantidades del segundo dataframe al primero, respetando los límites indicados en el campo Quantity del primer dataframe.
    df_pos: requiere las columnas 'PO', 'Modelo', 'Quantity' y 'Assigned'
    df_to_assign: requiere las columnas 'PO', 'Modelo', 'Quantity'
    """
    df_pos = df_pos.copy()
    df_to_assign=df_to_assign.copy()
    df_pos['Assigned'] = 0

    # Create a new DataFrame to track WorkOrder assignments
    df_assignments = []

    # Assign produced quantities respecting the limits
    for ln_index,ln_row in df_to_assign[['PO','Modelo']].drop_duplicates().iterrows():
        df_assign_subset = df_to_assign[(df_to_assign['PO'] == ln_row['PO']) & (df_to_assign['Modelo'] == ln_row['Modelo'])].copy()
        df_assign_subset['Remaining'] = df_assign_subset['Quantity']

        for wo_index, subset_row in df_assign_subset.iterrows():
            remaining_produced = subset_row['Remaining']

            for index, row in df_pos[(df_pos['PO'] == ln_row['PO']) & (df_pos['Modelo'] == ln_row['Modelo'])].iterrows():
                if remaining_produced <= 0:
                    break

                available_space = row['Quantity'] - df_pos.at[index, 'Assigned']
                assignable = min(available_space, remaining_produced)

                if assignable > 0:
                    df_pos.at[index, 'Assigned'] += assignable
                    remaining_produced -= assignable

                    # Track the assignment
                    line_assignment ={
                        # 'WO': subset_row['WO'],
                        'PO': row['PO'],
                        'Modelo': row['Modelo'],
                        'AvailQuantity': subset_row['Quantity'],
                        'LineNumber': row['LineNumber'],
                        'Assigned_Quantity': assignable
                        }
                    # Other fields if exist
                    if additional_fields:
                        for field in additional_fields:
                            if field in subset_row:
                                line_assignment[field]=subset_row[field]
                    df_assignments.append(line_assignment)

    # Create the assignments DataFrame
    df_assignments = pd.DataFrame(df_assignments)
    assigned={'df_pos':df_pos,'df_assignments':df_assignments}
    return assigned


In [30]:
# Reporte de work orders, que se encuentra en proceso de produccion
path_tracker=get_path(file_selectors,'Tracker')
df_wo_wb=pd.read_excel(path_tracker,sheet_name=None)
df_wo=pd.DataFrame()
for sheet in df_wo_wb.keys():
    if ('Plan de produccion' in sheet) or ('TERMINADAS' in sheet) :
        df_wo=pd.concat([df_wo,df_wo_wb[sheet]])
        df_wo.dropna(subset=['PO cliente','Modelo'],inplace=True)
check_mandatory_cols(df_wo.columns,'Tracker')
df_wo=standardize_columns(df_wo,column_mapping)
df_wo['Modelo']=df_wo['Modelo'].str.upper()
#% Edi
df_edi=rename_columns(df_edi,df_col_rel,table_from='ELP Master',sheet_from='EDI Master')
df_edi['Modelo']=df_edi['Modelo'].str.upper()

In [31]:
# Procesar envios al cliente eliminando cantidades negativas. Se conserva la fecha mas nueva de envio.
if not os.path.exists(output_paths['path_ship_cust']):
    show_popup_message('No hay envios al paso, integre al menos un Shipment Transactions')
    raise SystemExit()
df_ship_cust=read_excel(path=output_paths['path_ship_cust'])
df_ship_cust=standardize_columns(df_ship_cust,column_mapping)
df_ship_cust=rename_columns(df_ship_cust,df_col_rel,table_from='Shipment transactions')
df_ship_cust['Modelo']=df_ship_cust['Modelo'].str.upper()
df_ship_cust_grp=df_ship_cust.groupby(['PO','Modelo']).sum('Qty. Shipped').reset_index()
df_ship_cust_grp=df_ship_cust_grp[['PO','Modelo','Quantity']]
df_ship_cust_dates=df_ship_cust.sort_values(['PO','Modelo','ShipmentDateCust']).drop_duplicates(subset=['PO','Modelo'],keep='last')
df_ship_cust_dates.drop('Quantity',axis=1,inplace=True)
df_ship_cust_dates=df_ship_cust_dates.merge(df_ship_cust_grp,on=['PO','Modelo'])
df_ship_cust_dates=df_ship_cust_dates[['PO','Modelo','Quantity','ShipmentDateCust']]

In [32]:
# Envios a ELP
df_ship_elp=rename_columns(df_ship_elp,df_col_rel,table_from='ELP Master',sheet_from='Shipment to ELP')
df_ship_elp['Modelo']=df_ship_elp['Modelo'].str.upper()
df_ship_elp['PO']=df_ship_elp['PO'].str.strip()
df_ship_elp['Modelo']=df_ship_elp['Modelo'].str.strip()
df_ship_elp['Quantity'].fillna(0,inplace=True)
df_ship_elp_grp=df_ship_elp.groupby(['PO','Modelo','ShipmentDateELP']).sum('Quantity').reset_index()
df_ship_elp_grp=df_ship_elp_grp[['PO','Modelo','Quantity','ShipmentDateELP']]

In [33]:
# Merge EDI con work orders y embarques
assignments_wo=assign_quantities(df_pos=df_edi,df_to_assign=df_wo,additional_fields=['WO','START DATE','FINISH DATE','REPROGRAMACION','SHIP DATE'])
df_edi_combined=assignments_wo['df_pos']
df_edi_combined.rename({'Assigned':'WO Qty'},axis=1,inplace=True)
assignments_shp_cust=assign_quantities(df_pos=df_edi_combined,df_to_assign=df_ship_cust_dates,additional_fields=['ShipmentDateCust'])
df_edi_combined=assignments_shp_cust['df_pos']
df_edi_combined.rename({'Assigned':'Shipped to Cust'},axis=1,inplace=True)
assignments_shp_elp=assign_quantities(df_pos=df_edi_combined,df_to_assign=df_ship_elp,additional_fields=['ShipmentDateELP'])
df_edi_combined=assignments_shp_elp['df_pos']
df_edi_combined.rename({'Assigned':'Shipped to Elp'},axis=1,inplace=True)

### OOR Report

In [34]:
# Add additional fields to EDI, only the record with last ship date, the detail is in Work orders sheet, Shipped to ELP and Shipped to cust reports

def merge_additional_fields(df=pd.DataFrame(),df_edi=pd.DataFrame(),fields=[],sort_fields=[],key=[]):
    if len(df)==0:
        df=pd.DataFrame(columns=fields)
    df=df.sort_values(sort_fields)
    df.drop_duplicates(key,keep='last',inplace=True)
    df_edi=df_edi.merge(df[fields],how='left',on=key)
    return df_edi

df_assigned_wo = assignments_wo['df_assignments']
df_edi_combined=merge_additional_fields(df_assigned_wo,
                           df_edi_combined,
                           fields=['PO','Modelo','LineNumber', 'START DATE', 'FINISH DATE','SHIP DATE','REPROGRAMACION'],
                           sort_fields=['SHIP DATE'],
                           key=['PO','Modelo','LineNumber'])
# Add WO field
df_wo_qty=df_assigned_wo[['PO','Modelo','LineNumber','WO','AvailQuantity']]
df_wo_qty['WO']=df_wo_qty['WO'].astype(int).astype(str) + ' TotQty: ' + df_wo_qty['AvailQuantity'].astype(int).astype(str) + '.'
df_wo_qty=df_wo_qty.groupby(['PO','Modelo','LineNumber']).agg({
    'WO': ' '.join
}).reset_index()
df_edi_combined=merge_additional_fields(df_wo_qty,
                           df_edi_combined,
                           fields=['PO','Modelo','LineNumber', 'WO'],
                           sort_fields=['PO'],
                           key=['PO','Modelo','LineNumber'])


df_assigned_shp_elp=assignments_shp_elp['df_assignments']
df_edi_combined=merge_additional_fields(df_assigned_shp_elp,
                           df_edi_combined,
                           fields=['PO','Modelo','LineNumber','ShipmentDateELP'],
                           sort_fields=['ShipmentDateELP'],
                           key=['PO','Modelo','LineNumber'])
df_assigned_shp_cust=assignments_shp_cust['df_assignments']
df_edi_combined=merge_additional_fields(df_assigned_shp_cust,
                           df_edi_combined,
                           fields=['PO','Modelo','LineNumber','ShipmentDateCust'],
                           sort_fields=['ShipmentDateCust'],
                           key=['PO','Modelo','LineNumber'])

In [35]:
# Set production, and shipped STATUS
df_edi_combined[['Quantity','WO Qty','Shipped to Elp','Shipped to Cust']].fillna(0,inplace=True)
df_edi_combined['Quantity']=df_edi_combined['Quantity'].astype(float)
df_edi_combined.loc[(df_edi_combined['Quantity']-df_edi_combined['Shipped to Elp']<=0),'Movement Status']='Moved'
df_edi_combined.loc[(df_edi_combined['Quantity']-df_edi_combined['Shipped to Cust']<=0),'Status']='Shipped'
cols=['Quantity','WO Qty','Shipped to Elp','Shipped to Cust']
df_po_status_cls=df_edi_combined[['PO']+cols]
for col in cols:
    df_po_status_cls[col]=df_po_status_cls[col].astype(float)
df_po_status_cls=df_po_status_cls.groupby(['PO']).sum()
df_po_status_cls.reset_index(inplace=True)


df_po_status=df_po_status_cls[['PO','Quantity','WO Qty','Shipped to Elp','Shipped to Cust']]
df_edi_combined.sort_values(['PO','Modelo','LineNumber','PODate'],inplace=True)
df_edi_combined.reset_index(drop=True,inplace=True)
df_edi_combined['Balance to ship']=df_edi_combined['Quantity']-df_edi_combined['Shipped to Elp']

df_po_status_cls=df_po_status_cls.loc[(df_po_status_cls['Quantity']-df_po_status_cls['Shipped to Cust'])<=0,['PO']]
df_po_dates=df_assigned_shp_cust.sort_values(['PO','ShipmentDateCust']).drop_duplicates('PO',keep='last')[['PO','ShipmentDateCust']]
df_po_status_cls=df_po_status_cls.merge(df_po_dates,how='left',on=['PO'])
df_po_status_cls.rename(columns={'ShipmentDateCust':'\nPO closure date'},inplace=True)
df_edi_combined=df_edi_combined.merge(df_po_status_cls,how='left',on=['PO'])
df_edi_combined=set_family(df_edi_combined,column='Modelo',dest_col='Familia')
df_po_type=df_edi_combined[['PO','Familia']].drop_duplicates()
df_po_type.loc[df_po_type['Familia']=='Accesorio','PO TYPE']='Accesories only'
df_po_type.loc[(~(df_po_type['Familia']=='Accesorio'),'PO TYPE')]='Fixtures only'
df_po_type=df_po_type.drop_duplicates(['PO','PO TYPE']).sort_values(['PO','PO TYPE'])
df_po_type=df_po_type.groupby('PO', as_index=False).agg({'PO TYPE': '/'.join})
df_po_type.loc[df_po_type['PO TYPE']=='Accesories only/Fixtures only','PO TYPE']='Both types'
df_edi_combined=df_edi_combined.merge(df_po_type,how='left',on='PO')

df_edi_combined.loc[df_edi_combined['Order/Line cancelled?']=='Cancelled','Status']='Cancelled'

if not 'EDI Received' in df_edi_combined.columns:
    df_edi_combined['EDI Received']=''


In [36]:
# Read customer shipments and shipments to elp
df_ship_cust=read_excel(output_paths['path_ship_cust'])
df_ship_cust=standardize_columns(df_ship_cust,column_mapping)
df_ship_cust['Modelo']=df_ship_cust['Modelo'].str.upper()
df_ship_cust.sort_values(['PO','Modelo','Date Shipped'],inplace=True)
df_ship_cust.reset_index(drop=True,inplace=True)
# df_ship_elp=read_excel(output_paths['path_ship_elp'])
df_ship_elp=read_excel(path_ship_elp_master,sheet_name='Shipment to ELP')
df_ship_elp=rename_columns(df_ship_elp,df_col_rel,table_from='ELP Master',sheet_from='Shipment to ELP')
df_ship_elp['Modelo']=df_ship_elp['Modelo'].str.upper()
df_ship_elp.sort_values(['PO','Modelo','ShipmentDateELP'],inplace=True)
df_ship_elp.reset_index(drop=True,inplace=True)

In [37]:
# Get indexes to use in hyperlinks
def get_df_idx(df,idx_cols,idx_name='idx'):
    df_idx=df.reset_index()
    df_idx.rename({'index':idx_name},axis=1,inplace=True)
    df_idx=df_idx.drop_duplicates(idx_cols,keep='first')
    df_idx=df_idx[idx_cols+[idx_name]]
    return df_idx

df_wo.reset_index(inplace=True,drop=True)
df_ship_elp.reset_index(inplace=True,drop=True)
df_ship_cust.reset_index(inplace=True,drop=True)
df_wo_idx=get_df_idx(df_wo,idx_cols=['PO','Modelo'],idx_name='idx_wo')
df_ship_elp_idx=get_df_idx(df_ship_elp,idx_cols=['PO','Modelo'],idx_name='idx_elp')
df_ship_cust_idx=get_df_idx(df_ship_cust,idx_cols=['PO','Modelo'],idx_name='idx_cust')
df_edi_combined=df_edi_combined.merge(df_wo_idx,how='left',on=['PO','Modelo'])
df_edi_combined=df_edi_combined.merge(df_ship_elp_idx,how='left',on=['PO','Modelo'])
df_edi_combined=df_edi_combined.merge(df_ship_cust_idx,how='left',on=['PO','Modelo'])

In [38]:
# Generate hyperlinks
def set_hyperlink(df,sheet_name,col_name,idx_name,typ='str'):
    """
    sheet_name: Destination sheet of the hyperlink
    col_name: Column name where the hyperlink will be inserted
    idx_name: Column name where the index is located
    """
    df=df.copy()
    idx=~df[idx_name].isnull()
    if typ=='str':
        quote='\"'
    else:
        quote=''
    df.loc[idx,col_name]=f"=HYPERLINK(\"#'{sheet_name}'!A"+(df.loc[idx,idx_name]+2).astype(int).astype(str)+f"\",{quote}"+(df.loc[idx,col_name]).astype(str)+f"{quote})"
    return df
df_edi_combined=set_hyperlink(df_edi_combined,sheet_name='WorkOrder Detail',col_name='WO',idx_name='idx_wo')
df_edi_combined=set_hyperlink(df_edi_combined,sheet_name='Shipped to Cust',col_name='Shipped to Cust',idx_name='idx_cust',typ='int')
df_edi_combined=set_hyperlink(df_edi_combined,sheet_name='Shipped to Elp',col_name='Shipped to Elp',idx_name='idx_elp',typ='int')

### Actualizar OOR con datos nuevos

In [39]:
# Actualizar OOR
df_oor=rename_columns(df_edi_combined,df_col_rel,table_from='EDI Combined',table_to='OOR Report',sheet_to='OOR')

df_oor=set_family(df_oor,'ProductServiceID')
oor_cols=df_columns[df_columns['table']=='OOR Report']['column_name'].to_list()
# Calculo de valores por columna
df_oor=format_dates(df_oor,['START DATE','FINISH DATE','SHIP DATE','EDI Received','POUS Date','Actual Ship date to Customer\n','Actual Move date from CUU'],type='')
df_oor.loc[df_oor['AssignedDropZone']=='','AssignedDropZone']='NULL'
df_oor['AssignedDropZone'].fillna('NULL',inplace=True)
df_oor['NP+PO+DZ']=df_oor['ProductServiceID']+df_oor['PurchaseOrder']+df_oor['AssignedDropZone'].astype(str)
df_oor['CONCAT\nPN+PO']=df_oor['ProductServiceID']+df_oor['PurchaseOrder']

oor_current_cols=[x for x in oor_cols if x in df_oor]

df_oor=df_oor[oor_current_cols]
df_oor.sort_values(['Family ','PurchaseOrder','ProductServiceID','LineNumber'],inplace=True)
df_oor['PO Aging']='' # Formula
df_oor['Aging Category']='' # Formula
df_oor['Balance to move']='' # Formula
# Filtrar segun la seleccion de fecha
df_oor=df_oor[df_oor['POUS Date'] >= pd.to_datetime(datepicker_freeze.value)]

#Integrar datos al OOR Anterior
path_oor_old=get_path(file_selectors,'OOR')
wb_oor_old=load_workbook(path_oor_old)
ws_oor_old=wb_oor_old['OOR']
dict_oor_old=get_worksheed_df(ws_oor_old,'Family')
check_mandatory_cols(dict_oor_old['df'].columns,'OOR')
# Part of the OOR that will be updated, we will remove duplicates from this part
df_oor_to_update=dict_oor_old['df'][dict_oor_old['df']['POUS Date'] >= pd.to_datetime(datepicker_freeze.value)]
# This part will be kept as is, duplicates are allowed
dict_oor_old['df']=dict_oor_old['df'][dict_oor_old['df']['POUS Date'] < pd.to_datetime(datepicker_freeze.value)]

key_cols=['PurchaseOrder','ProductServiceID','LineNumber','AssignedDropZone']
df_oor_to_update.drop_duplicates(subset=key_cols,inplace=True)
df_oor_to_update=update_dataframe(df_oor_to_update,df_oor,key_cols)
dict_oor_old['df']=pd.concat([dict_oor_old['df'],df_oor_to_update])
dict_oor_old['df'].reset_index(drop=True,inplace=True)

ws_oor_old=update_sheet(dict_oor_old,ws_oor_old)

### Formato de excel

In [40]:
# Formato excel
def move_columns_to_front(df, columns):
    """
    Move specified columns to the beginning of the DataFrame.
    """
    columns = [col for col in columns if col in df.columns]
    remaining_columns = [col for col in df.columns if col not in columns]
    return df[columns + remaining_columns]

close_xl_if_open(get_path(file_selectors,'OOR'))
df_wo=format_dates(df_wo,['Date','START DATE', 'FINISH DATE', 'REPROGRAMACION','SHIP DATE'])
df_wo=move_columns_to_front(df_wo,['PO','Modelo','WO','Quantity','START DATE', 'FINISH DATE', 'REPROGRAMACION','SHIP DATE'])
df_ship_elp.loc[df_ship_elp['DZ']=='','DZ']='NULL'
df_ship_elp['DZ'].fillna('NULL',inplace=True)
df_ship_elp=format_dates(df_ship_elp,['ShipmentDate'])
df_ship_elp=move_columns_to_front(df_ship_elp,['PO','Modelo','ShipmentDate','Quantity'])
df_ship_cust=format_dates(df_ship_cust,['Date Shipped'])
df_ship_cust=move_columns_to_front(df_ship_cust,['PO','Modelo','Date Shipped','Qty. Shipped'])
wb_oor={}

wb_oor_old=create_new_sheet(wb_oor_old,sheet_name='WorkOrder Detail',df=df_wo)
wb_oor_old=create_new_sheet(wb_oor_old,sheet_name='Shipped to Elp',df=df_ship_elp)
wb_oor_old=create_new_sheet(wb_oor_old,sheet_name='Shipped to Cust',df=df_ship_cust)

#Apply autofilter and freeze panes
wb_oor_old['WorkOrder Detail'].auto_filter.ref = wb_oor_old['WorkOrder Detail'].dimensions
wb_oor_old['WorkOrder Detail'].freeze_panes = 'A2'
wb_oor_old['Shipped to Elp'].auto_filter.ref = wb_oor_old['Shipped to Elp'].dimensions
wb_oor_old['Shipped to Elp'].freeze_panes = 'A2'
wb_oor_old['Shipped to Cust'].auto_filter.ref = wb_oor_old['Shipped to Cust'].dimensions
wb_oor_old['Shipped to Cust'].freeze_panes = 'A2'

# Format row when key values change
dict_special_formats=get_xl_formatting('special_format')
col_info1=get_column_info(ws_oor_old,'PurchaseOrder')['data_range']
col_info2=get_column_info(ws_oor_old,'ProductServiceID')['data_range']
format_on_change(zip(col_info1,col_info2),ws_oor_old,start_row=1,format1=dict_special_formats['on_change_a'],format2=dict_special_formats['on_change_b'])

# Format dates
wb_oor_old=format_xl_dates(wb_oor_old,sheet_name='OOR',date_columns=['POUS Date',
                                                     'EDI Received',
                                                     '\nPO closure date',
                                                     'Actual Move date from CUU',
                                                     'Actual Ship date to Customer\n',
                                                     'Estimated \nMOVE DATE\nCUU',
                                                     'Reprogrammed finish date CUU',
                                                     'START DATE',
                                                     'FINISH DATE',
                                                     'Actual Ship date to Customer\n'])


ws=wb_oor_old['WorkOrder Detail']
col_info1=get_column_info(ws,'PO')['data_range']
col_info2=get_column_info(ws,'Modelo')['data_range']
format_on_change(zip(col_info1,col_info2),ws,start_row=1,format1=dict_special_formats['on_change_a'],format2=dict_special_formats['on_change_b'])


ws=wb_oor_old['Shipped to Elp']
col_info1=get_column_info(ws,'PO')['data_range']
col_info2=get_column_info(ws,'Modelo')['data_range']
format_on_change(zip(col_info1,col_info2),ws,start_row=1,format1=dict_special_formats['on_change_a'],format2=dict_special_formats['on_change_b'])

ws=wb_oor_old['Shipped to Cust']
col_info1=get_column_info(ws,'PO')['data_range']
col_info2=get_column_info(ws,'Modelo')['data_range']
format_on_change(zip(col_info1,col_info2),ws,start_row=1,format1=dict_special_formats['on_change_a'],format2=dict_special_formats['on_change_b'])

# Apply column sizes
# if col_sizes:
#     wb=apply_col_sizes(wb,col_sizes)

save_wb(wb_oor_old,path_oor_old)
os.startfile(path_oor_old)

# AAR

In [ ]:
#Yield reports
def map_yield_report(df):
    df_yield=df.copy()
    df_yield.columns=df_yield.iloc[8].fillna('NA').to_list()
    df['Yield Reports'].fillna('',inplace=True)
    date_from=df[df['Yield Reports'].str.contains('From')].iloc[0]['Yield Reports']
    date_to=df[df['Yield Reports'].str.contains('From')].iloc[0]['Unnamed: 5']
    df_yield['date_from']=date_from
    df_yield['date_to']=date_to
    df_yield=df_yield[(~df_yield['Part Number'].isna()) & (df_yield['Part Number']!='Part Number')]
    df_yield['date_from']=df_yield['date_from'].str.replace('From: ','')
    df_yield['date_to']=df_yield['date_to'].str.replace('To: ','')
    df_yield['date_from']=pd.to_datetime(df_yield['date_from'])
    df_yield['date_to']=pd.to_datetime(df_yield['date_to'])
    df_yield.drop('NA',axis=1,inplace=True)
    df_yield.fillna(0,inplace=True)
    # Main date is date_from
    df_yield['month']=df_yield['date_from'].dt.strftime('%Y-%m')
    return df_yield

yield_files_lst=os.listdir(folder_yield_label.value)
df_yield=pd.DataFrame()
for file in yield_files_lst:
    df=read_excel(os.path.join(folder_yield_label.value,file))
    df=map_yield_report(df)
    df['file_name']=file
    df_yield=pd.concat([df,df_yield])


dict_xl_fields={"ENSAMBLE DE BISAGRA Y TAPAS":"Complete mechanical Assy output",
 "ANGLE/DIMMING TEST":"Angle/Dimming test output",
#  "DIMMING TEST":"Angle/Dimming test output",
 "ENSAMBLE DEL BRACKET":"Mechanical Assy"}

for process in df_yield['Process'].drop_duplicates():
    if not process in dict_xl_fields.keys():
        continue
    df_yield.loc[df_yield['Process']==process,'xl_field']=dict_xl_fields[process]


df_yield=df_yield.groupby(['date_from','xl_field']).sum('Total Tested').reset_index()
df_yield=df_yield[['date_from','xl_field','Total Tested']]
df_yield['date_from']=pd.to_datetime(df_yield['date_from']).dt.normalize()
# Llenar el formato
search_parms={
    "column":'date_from',
    "row":'xl_field',
    "offset":
            {"Category":
            {
            "scheduled":(0,0),
            "real":(1,0)
            },
            "Shift":
            {"1s":(0,0),
            "2s":(0,2)},
            }
}

def fill_yield_report(df,wb,sheet_name='',search_parms=search_parms):
    ws=wb[sheet_name]
    df_data_coord=df[[search_parms['column'],search_parms['row']]].drop_duplicates()
    df_data_coord['row']=''
    df_data_coord['column']=''
    coord_y=get_locations(df_data_coord,ws,search_parms['row'])
    coord_x=get_locations(df_data_coord,ws,search_parms['column'])


    for index,row in df.iterrows():
        if not ((coord_x[row[search_parms['column']]] and coord_y[row[search_parms['row']]])):
            continue  
        cell=ws.cell(ws[coord_y[row[search_parms['row']]]].row,
                    ws[coord_x[row[search_parms['column']]]].column)
        for key in search_parms['offset'].keys():
            if not key in df.columns:
                continue
            offset=search_parms['offset'][key][row[key]]
            cell=cell.offset(offset[0],offset[1])
        cell.value=row['Total Tested']
    return wb

wb=load_workbook(path_oor_old)


wb=fill_yield_report(df_yield,wb,sheet_name='Trov Daily Status')
wb=fill_yield_report(df_yield,wb,sheet_name='Rise Daily Status')
wb.save(path_oor_old)
os.startfile(path_oor_old)